# Exercise 3.4: RDataFrame versus the explicit loop revised

From *Programming in High Energy Particle Physics*, Chapter 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch03/ex03_4_solution.ipynb)

Exercise 3.4 RDataFrame versus the explicit loop revised Implement the dilepton-mass analysis of Exercise 3.1 twice: once with an explicit event loop as in Listing 3.7 , once with RDataFrame. Compare the code length and readability. Benchmark the execution time with and without ROOT::EnableImplicitMT() on a tree with 100,000+ events. Report the speedup factor.

<details><summary>Hint</summary>

Use TStopwatch around the first access to a result, and repeat each measurement a few times because the first run also includes JIT compilation and cold file caches. A tree of 100,000 small events may be too small to benefit from many threads: try increasing the number of events in Listing 3.6 to a few million.

</details>

**Colab setup:** ROOT installation takes about 3–5 minutes and restarts the runtime once. After that restart, run all cells from the top again. The setup cells are safe to rerun.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q condacolab")
        import condacolab
        condacolab.install()  # one-time runtime restart

In [ ]:
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().system("mamba install -q -y -c conda-forge root")
        import ROOT
else:
    import ROOT
ROOT.gROOT.SetBatch(True)

### Generate events

Use 100,000 or more for a meaningful MT benchmark; the default is 8,000 to fit a short Colab CPU run.

In [ ]:
import math
from array import array
from pathlib import Path
def create_tree(n_events=8000):
    rng=ROOT.TRandom3(42)
    f=ROOT.TFile("events.root","RECREATE")
    tree=ROOT.TTree("events","Toy dilepton events")
    floats={name:ROOT.std.vector('float')() for name in
        ('electron_pt','electron_eta','electron_phi','electron_iso',
         'muon_pt','muon_eta','muon_phi','muon_iso')}
    ints={name:ROOT.std.vector('int')() for name in ('electron_charge','muon_charge')}
    has_Z=array('i',[0]);z_pass=array('i',[0]);y_Z=array('f',[0.])
    for name,v in {**floats,**ints}.items():tree.Branch(name,v)
    tree.Branch('has_Z',has_Z,'has_Z/I');tree.Branch('z_pass',z_pass,'z_pass/I')
    tree.Branch('y_Z',y_Z,'y_Z/F')
    for _ in range(n_events):
        for v in [*floats.values(),*ints.values()]:v.clear()
        has_Z[0]=int(rng.Rndm()<0.35);z_pass[0]=0
        y_Z[0]=rng.Gaus(0,1.3) if has_Z[0] else 0.0
        for flavor in ('electron','muon'):
            for _ in range(rng.Integer(3)):
                pt=rng.Exp(25)+5;eta=rng.Gaus(0,1.8)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(rng.Uniform(-math.pi,math.pi))
                floats[flavor+'_iso'].push_back(rng.Exp(0.25))
                ints[flavor+'_charge'].push_back(1 if rng.Rndm()<0.5 else -1)
        if has_Z[0]:
            flavor='electron' if rng.Rndm()<0.5 else 'muon'
            mass=max(70.,rng.Gaus(91.1876,2.8))
            costh=rng.Uniform(-0.98,0.98);eta_star=math.atanh(costh)
            pt_true=0.5*mass*math.sqrt(1-costh*costh)
            z_pass[0]=int(pt_true>20 and abs(y_Z[0]+eta_star)<2.4
                          and abs(y_Z[0]-eta_star)<2.4)
            phi=rng.Uniform(-math.pi,math.pi)
            for charge,eta,azimuth in ((1,y_Z[0]+eta_star,phi),
                                       (-1,y_Z[0]-eta_star,phi+math.pi)):
                pt=pt_true*rng.Gaus(1,0.02)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(azimuth)
                floats[flavor+'_iso'].push_back(rng.Exp(0.12/(1+pt/50)))
                ints[flavor+'_charge'].push_back(charge)
        tree.Fill()
    tree.Write();f.Close()
    print(f"Created {n_events} toy events in events.root")
N_EVENTS=8000  # set to 50_000 for the full exercise
create_tree(N_EVENTS)

### Repeat timings and compare results

In [ ]:
import statistics
ROOT.gInterpreter.Declare(r"""#ifndef HEP_NOTEBOOK_PAIR_MASSES
#define HEP_NOTEBOOK_PAIR_MASSES
#include <ROOT/RVec.hxx>
#include <Math/Vector4D.h>
ROOT::VecOps::RVec<float> pair_masses(const ROOT::VecOps::RVec<float>& pt,
 const ROOT::VecOps::RVec<float>& eta,const ROOT::VecOps::RVec<float>& phi,
 const ROOT::VecOps::RVec<int>& charge,float lep_mass){
 ROOT::VecOps::RVec<float> masses;
 for(size_t i=0;i<pt.size();++i)for(size_t j=i+1;j<pt.size();++j){
  if(charge[i]*charge[j]>=0)continue;
  ROOT::Math::PtEtaPhiMVector a(pt[i],eta[i],phi[i],lep_mass);
  ROOT::Math::PtEtaPhiMVector b(pt[j],eta[j],phi[j],lep_mass);
  masses.push_back((a+b).M());
 }
 return masses;
}
#endif""")
def explicit_loop():
    f=ROOT.TFile.Open("events.root");tree=f.Get("events")
    hist=ROOT.TH1D("h_explicit",";m_{ll} [GeV];Pairs",60,60,120)
    hist.SetDirectory(0)
    for ev in tree:
        for flavor,mass in (("electron",0.000511),("muon",0.105658)):
            pt=getattr(ev,flavor+"_pt");eta=getattr(ev,flavor+"_eta")
            phi=getattr(ev,flavor+"_phi");charge=getattr(ev,flavor+"_charge")
            for i in range(len(pt)):
                for j in range(i+1,len(pt)):
                    if charge[i]*charge[j]>=0:continue
                    a=ROOT.Math.PtEtaPhiMVector(pt[i],eta[i],phi[i],mass)
                    b=ROOT.Math.PtEtaPhiMVector(pt[j],eta[j],phi[j],mass)
                    hist.Fill((a+b).M())
    f.Close();return hist.GetEntries()
def rdf_loop(trial):
    df=ROOT.RDataFrame("events","events.root")
    df=df.Define("ee","pair_masses(electron_pt,electron_eta,electron_phi,electron_charge,0.000511f)")
    df=df.Define("mm","pair_masses(muon_pt,muon_eta,muon_phi,muon_charge,0.105658f)")
    he=df.Histo1D((f"he{trial}","",60,60,120),"ee")
    hm=df.Histo1D((f"hm{trial}","",60,60,120),"mm")
    return he.GetValue().GetEntries()+hm.GetValue().GetEntries()
def timed(fn):
    sw=ROOT.TStopwatch();sw.Start();result=fn();sw.Stop()
    return sw.RealTime(),result
explicit_times=[];serial_times=[];mt_times=[]
for trial in range(3):
    t,n1=timed(explicit_loop);explicit_times.append(t)
    t,n2=timed(lambda:rdf_loop(trial));serial_times.append(t)
    assert n1==n2
ROOT.EnableImplicitMT(2)
for trial in range(3,6):
    t,n3=timed(lambda:rdf_loop(trial));mt_times.append(t)
ROOT.DisableImplicitMT()
print("Pairs:",n1)
print("Median times [s]: explicit",statistics.median(explicit_times),
      "RDF serial",statistics.median(serial_times),"RDF 2-thread",statistics.median(mt_times))
print("MT/serial speedup:",statistics.median(serial_times)/statistics.median(mt_times))

In [ ]:
assert n1>0 and n1==n3

### Interpretation

The notebook prints the measured speedup for the active Colab runtime. On the default small tree, two threads may be slower than serial because dispatch and JIT overhead dominate. The Python explicit loop also includes Python/C++ boundary overhead, which matters when interpreting that comparison.